In [1]:
pip install langchain-google-genai langchain_community pymupdf langchain-text-splitters faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [64]:
import os
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate # Prompt
from langchain_google_genai import ChatGoogleGenerativeAI # Lanchain Google GeminiAI
from langchain_core.output_parsers import StrOutputParser # Output Format
from langchain_community.document_loaders import PyMuPDFLoader # Extract the Data from PDF
from langchain_text_splitters import RecursiveCharacterTextSplitter # Chunk Data
from langchain_google_genai import GoogleGenerativeAIEmbeddings # Gemini Embedding
from langchain_community.vectorstores import FAISS # Vector DB
from langchain_core.runnables import RunnablePassthrough # for passing the user question to a retireve as well as the prompt (both)


In [65]:
os.environ['GOOGLE_API_KEY'] = userdata.get('RAG_API')

In [66]:
model_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",
                                      temperature = 1)

In [67]:
# Step1 - Data Extraction

file_path = "/content/SBI Incurance.pdf"
loader = PyMuPDFLoader(file_path)
docs = loader.load()

In [68]:
# this is the entire data, but we will not want this entire data component to be going into a model (LLM) as that will cost more input tokens and can have hallucination
docs

[Document(metadata={'producer': 'Nitro PDF PrimoPDF', 'creator': 'PrimoPDF http://www.primopdf.com', 'creationdate': '2020-07-16T18:50:21-05:30', 'source': '/content/SBI Incurance.pdf', 'file_path': '/content/SBI Incurance.pdf', 'total_pages': 27, 'format': 'PDF 1.3', 'title': 'Microsoft Word - Health Insurance Policy -Retail', 'author': 'user', 'subject': '', 'keywords': '', 'moddate': '2020-07-16T18:50:21-05:30', 'trapped': '', 'modDate': "D:20200716185021-05'30'", 'creationDate': "D:20200716185021-05'30'", 'page': 0}, page_content='SBI General Insurance Company \nSBI General Insurance Company \nSBI General Insurance Company \nSBI General Insurance Company Limited\nLimited\nLimited\nLimited\n                 \n \nSBI General Insurance Company Limited  \nSBI General Insurance Company Limited  \nSBI General Insurance Company Limited  \nSBI General Insurance Company Limited   \nCorporate & Registered Office:   \'Natraj\', 301, Junction of  Western Express Highway & Andheri - Kurla Road,

In [69]:
# Step2 - Chunking (splitting documents into smaller components)
text_spitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 300,
    add_start_index = True
)

chunks = text_spitter.split_documents(docs)

In [70]:
chunks

[Document(metadata={'producer': 'Nitro PDF PrimoPDF', 'creator': 'PrimoPDF http://www.primopdf.com', 'creationdate': '2020-07-16T18:50:21-05:30', 'source': '/content/SBI Incurance.pdf', 'file_path': '/content/SBI Incurance.pdf', 'total_pages': 27, 'format': 'PDF 1.3', 'title': 'Microsoft Word - Health Insurance Policy -Retail', 'author': 'user', 'subject': '', 'keywords': '', 'moddate': '2020-07-16T18:50:21-05:30', 'trapped': '', 'modDate': "D:20200716185021-05'30'", 'creationDate': "D:20200716185021-05'30'", 'page': 0, 'start_index': 0}, page_content="SBI General Insurance Company \nSBI General Insurance Company \nSBI General Insurance Company \nSBI General Insurance Company Limited\nLimited\nLimited\nLimited\n                 \n \nSBI General Insurance Company Limited  \nSBI General Insurance Company Limited  \nSBI General Insurance Company Limited  \nSBI General Insurance Company Limited   \nCorporate & Registered Office:   'Natraj', 301, Junction of  Western Express Highway & Andhe

In [71]:
len(chunks)

81

In [72]:
# Step3 - Embedding

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [73]:
# Step4 - Vector DB which is FAISS

vectordb = FAISS.from_documents(chunks,embeddings)

In [74]:
vectordb

In [75]:
# Step5 - Retrieve. All vector databases and indexes have retrieve functionality

retriever = vectordb.as_retriever(search_kwargs = {'k':5}) #retriever is now initiated

# We will pass the user question to retriever which retrieves the 5 closest chunks
# and those 5 chuncks (append page content) and use it as context in prompt

In [76]:
# This function just appends 5 retrieved sections and make them a single text

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [77]:
template = """

You are a SBI Insurance guy bot which only answers questions related to Health Insurance policy using the following context:

{context}

Question: {user_question}

Additional instructions:
Do not answer anything outside the context provided
If questions are not related to the context, politely respond that you are not able to answer the question and reach out to SBI Insurance support over an email.

"""

prompt = ChatPromptTemplate.from_template(template)

In [78]:
parser = StrOutputParser()

In [79]:
# Final RAG chain - retriever that gives context from user question, prompt, model, outputformat

rag_overall = ( {'context':retriever | format_docs, 'user_question':RunnablePassthrough()} | prompt | model_gemini | parser)

# Retriever will first receive the user question with 'user_question' as variable
# Then retriver gets the 5 closer chunks
# Then format_docs will append all those 5 chunks and finally save as 'context' variable

# Then prompt gets created using the "user_question" intially provided and "context" created
# Then the model and parser are kind of self explainatory

In [80]:
input = 'what could be cover the Health Insurance?'
response = rag_overall.invoke(input)
print(response)

Under this Health Insurance policy, the following hospitalization expenses are covered, subject to the terms, conditions, exclusions, definitions, and sub-limits stipulated in the policy contract:

*   Room, Boarding Expenses
*   Medical Practitioners fees
*   Intensive Care Unit
*   Nursing Expenses
*   Surgical fees, operating theatre, Anesthetist, Anesthesia, Blood, Oxygen and their administration
*   Physio therapy while being treated as inpatient and being part of the treatment
*   Drugs and medicines consumed during hospitalization period
*   Hospital miscellaneous services (such as laboratory, X-ray, diagnostic tests)
*   Dressing, ordinary splints and plaster casts
*   Cost of Prosthetic devices if implanted during a surgical procedure


In [82]:
input = 'what scope of cover and tell me payments it will cover in the Health Insurance?'
response = rag_overall.invoke(input)
print(response)

As an SBI Insurance bot, I can help you understand the scope of cover and payments under your Health Insurance policy based on the information you've provided.

The Insurer will pay for expenses reasonably and necessarily incurred by or on behalf of the Insured Person, up to the Sum Insured and subject to any excess as specified in the policy schedule.

The following benefits are covered:

*   **Room, Board & Nursing expenses:** Covered up to 1% of the Sum Insured per day. If admitted to an Intensive Care Unit (ICU), it's covered up to 2% of the Sum Insured per day. However, all admissible claims under Room, Board & Nursing Expenses, including ICU, are restricted to a maximum of 25% of the Sum Insured per illness/injury during the policy period.
*   **Medical Practitioner, Surgeon, Anesthetist, Consultants, and Specialists Fees:** All admissible claims under this section are restricted to a maximum of 40% of the Sum Insured per illness/injury during the policy period.
*   **Anesthesia,

In [61]:
input = 'what could be cover the Accident Insurance Policy?'
response = rag_overall.invoke(input)
print(response)

I can help you with questions related to health insurance policies offered by SBI Insurance, based on the provided context.

Regarding your question about what an Accident Insurance Policy could cover, my current context is specifically about **Health Insurance**. Therefore, I am unable to provide information on Accident Insurance policies.

For inquiries about Accident Insurance, please reach out to SBI Insurance support over an email.


In [63]:
input = 'what kind of Diseases cover in Health Insurance Policy?'
response = rag_overall.invoke(input)
print(response)

As an SBI Insurance bot, I can only provide information based on the context you've given me. The provided text primarily details exclusions from coverage. It does not list the specific diseases that are covered by the Health Insurance Policy.

Therefore, I am unable to answer your question about what kind of diseases are covered. For detailed information on covered diseases, please reach out to SBI Insurance support over an email.


In [83]:
input = 'Treatment what Diseases cover in Health Insurance Policy?'
response = rag_overall.invoke(input)
print(response)

Under the SBI Health Insurance Policy, coverage is provided for medical treatment of diseases, illnesses, or injuries that require inpatient hospitalization. This includes costs associated with being a patient in a Hospital, Nursing Home, or Day Care Centre. The policy also covers the costs of a bed, treatment and care by medical staff, medical procedures, Medical Practitioner's fees, and medicines and consumables, including pacemakers and implants, if recommended by the attending Medical Practitioner.

However, there are certain exclusions based on the duration of the policy. For example, medical expenses for diseases/illnesses diagnosable within the first 30 days of the policy commencement are excluded, except for those resulting from accidental bodily injury. Certain specific diseases and surgeries are also excluded for the first year, first two years, and first three years of the policy, unless specific conditions for waiver are met, such as continuous coverage from another insurer